**Fig. 3, step 1: decompose the 11 ground-truth stacks into dense 2D slices and write the four split regimes (permissive and conservative, 7 and 11 stacks), lateral and axial**

In [ ]:
# Working directory: data/fig3/ under the repository root, where the Fig. 3 deposit folder is downloaded
# (decomposed_2D/, bootstrap_*_models/, domain-specialized_*/, benchmark_outputs*/, metrics_outputs/).
# The 11 ground-truth stacks are read from ../ground_truth/zebrafish/{development,held_out}/.
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # the notebook's own folder, for benchmarking_scripts/
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig3")
print("working directory:", os.getcwd())


In [ ]:
try:
    from cellpose import models, io, train, core
    from natsort import natsorted
    import tifffile as tiff
    print('Modules imported successfully')
except RuntimeError as e:
    print(f"Error importing cellpose: {e}")
    print("This might be due to a numpy version mismatch. Try restarting the runtime and running the cells again.")
except ImportError as e:
    print(f"Error importing a required library: {e}")
    print("Please ensure all necessary libraries are installed.")

Modules imported successfully


In [ ]:
import numpy as np
import tifffile
from skimage.segmentation import find_boundaries
from skimage.measure import label, regionprops
from skimage.morphology import opening, closing, disk
from typing import Tuple, List
import tqdm
import json


In [ ]:
import os
import time
from datetime import timedelta
import matplotlib.pyplot as plt
import numpy as np
from cellpose import models, train


In [ ]:
# Loader
class LoadTrainingData:
    """One ground-truth stack decomposed into the 2D slices of one view, with a train /
    validation / test split over those slices (Methods §3.2-3.4).

    Loading transposes the stack to the requested view (XY, XZ or YZ), drops `edge_buffer`
    slices at both ends of an axial view, and cleans every slice's instance labels when
    `relabel_masks=True` (`get_cleaned_mask_stack`). `split_data` then assigns slice indices to
    the three splits; `get_training` / `get_validation` / `get_test_data` return
    (images, masks, indices) lists. `save_data` / `load_saved_splits` persist the split indices;
    `plot_split_map` and `evaluate_model_slicewise_metrics` are inspection helpers.
    """
    def __init__(
        self,
        stack_name,
        image_path,
        mask_path,
        view="XY",
        train_split=0.7,
        validation_split=0.2,
        test_split=0.1,
        segment_range=None,
        save_path=None,
        load_saved=False,
        spatial_split=True,
        relabel_masks=True,
        size_filter=100,
        smoothing_disk_size=1,
        edge_buffer=0
    ):
        self.stack_name = stack_name
        self.image_path = image_path
        self.mask_path = mask_path
        self.segment_range = segment_range
        self.view = view.upper()
        self.train_split = train_split
        self.validation_split = validation_split
        self.test_split = test_split
        self.save_path = save_path
        self.spatial_split = spatial_split
        self.relabel_masks = relabel_masks
        self.size_filter = size_filter
        self.smoothing_disk_size = smoothing_disk_size
        self.edge_buffer = edge_buffer

        self.images, self.masks, self.slice_indices = self.load_data()

        if load_saved:
            self.load_saved_splits()
        else:
            self.train, self.validation, self.test = self.split_data(spatial_split=self.spatial_split)

    def load_data(self):
        """Read the image and mask TIFFs, transpose to `self.view`, apply the axial edge buffer
        and the per-slice label clean-up, and return (image slices, mask slices, slice infos).

        Each image slice is (C, H, W) with two channels; each info is
        {"stack", "view", "slice_idx"}.
        """
        import tifffile

        image = tifffile.imread(self.image_path)  # (Z, C, X, Y)
        mask = tifffile.imread(self.mask_path)    # (Z, X, Y)

        print(f"Loaded image shape: {image.shape}")
        print(f"Loaded mask shape: {mask.shape}")

        if self.segment_range:
            start, end = self.segment_range
            image = image[start:end]
            mask = mask[start:end]

        if image.shape[0] != 2 and image.shape[1] == 2:
            image = np.transpose(image, (1, 0, 2, 3))

        if mask.ndim == 4 and mask.shape[0] == 1:
            mask = np.squeeze(mask, axis=0)
        elif mask.ndim == 4 and mask.shape[-1] == 1:
            mask = np.squeeze(mask, axis=-1)

        if self.view == "XY":
            pass
        elif self.view == "YZ":
            mask = np.transpose(mask, (2, 0, 1))
            image = np.transpose(image, (0, 3, 1, 2))
        elif self.view == "XZ":
            mask = np.transpose(mask, (1, 0, 2))
            image = np.transpose(image, (0, 2, 1, 3))
        else:
            raise ValueError(f"Unsupported view: {self.view}")

        # Apply edge buffer
        if self.edge_buffer > 0 and self.view in ["XZ", "YZ"]:
            total_slices = mask.shape[0]
            start = self.edge_buffer
            end = total_slices - self.edge_buffer
            if end <= start:
                raise ValueError(f"Edge buffer too large: no slices left. Total: {total_slices}, Buffer: {self.edge_buffer}")
            mask = mask[start:end]
            image = image[:, start:end]

        if self.relabel_masks:
            print(f"Cleaning 2D instance labels in {self.view} view...")
            mask = self.get_cleaned_mask_stack(mask, min_size=self.size_filter, smoothing_disk_size=self.smoothing_disk_size)

        self.masks = mask
        self.images = image


        
        image_slices, mask_slices, infos = [], [], []
        for i in tqdm.tqdm(range(mask.shape[0]), desc=f"Extracting {self.view} slices"):
            image_slices.append(image[:, i])
            mask_slices.append(mask[i])
            infos.append({
                "stack": self.stack_name,
                "view": self.view,
                "slice_idx": i
            })
        
        return image_slices, mask_slices, infos





    def split_data(self, spatial_split="chunked", chunk_size=10):
        """
        Splits the dataset into train/val/test sets using different strategies.

        Args:
            spatial_split (str): One of ['random', 'contiguous', 'chunked']
            chunk_size (int): Used only if spatial_split='chunked'

        Returns:
            train, validation, test: lists of (img, mask, idx) tuples
        """
        num = len(self.images)
        indices = np.arange(num)

        if spatial_split == "random":
            np.random.shuffle(indices)
            n_train = int(self.train_split * num)
            n_val = int(self.validation_split * num)
            n_test = num - n_train - n_val

            train_idx = indices[:n_train]
            val_idx = indices[n_train:n_train + n_val]
            test_idx = indices[n_train + n_val:]

        elif spatial_split == "contiguous":
            n_train = int(self.train_split * num)
            n_val = int(self.validation_split * num)
            n_test = num - n_train - n_val

            train_idx = indices[:n_train]
            val_idx = indices[n_train:n_train + n_val]
            test_idx = indices[n_train + n_val:]

        elif spatial_split == "chunked":
            train_idx, val_idx, test_idx = [], [], []

            for start in range(0, num, chunk_size):
                end = min(start + chunk_size, num)
                chunk = indices[start:end]

                n = len(chunk)
                n_train = int(self.train_split * n)
                n_val = int(self.validation_split * n)
                n_test = n - n_train - n_val

                # Optional: shuffle within chunk
                chunk = chunk.copy()
                np.random.shuffle(chunk)

                train_idx.extend(chunk[:n_train])
                val_idx.extend(chunk[n_train:n_train + n_val])
                test_idx.extend(chunk[n_train + n_val:])

        else:
            raise ValueError(f"Invalid spatial_split option: {spatial_split}. Choose from 'random', 'contiguous', 'chunked'.")

        # Final assignment
        train = [(self.images[i], self.masks[i], self.slice_indices[i]) for i in train_idx]
        validation = [(self.images[i], self.masks[i], self.slice_indices[i]) for i in val_idx]
        test = [(self.images[i], self.masks[i], self.slice_indices[i]) for i in test_idx]

        return train, validation, test


    def get_training(self):
        """Return the training split as three lists: images, masks, slice infos.
        """
        imgs, masks, idxs = zip(*self.train)
        return list(imgs), list(masks), list(idxs)

    def get_validation(self):
        """Return the validation split as three lists: images, masks, slice infos.
        """
        imgs, masks, idxs = zip(*self.validation)
        return list(imgs), list(masks), list(idxs)

    def get_test_data(self):
        """Return the test split as three lists: images, masks, slice infos.
        """
        imgs, masks, idxs = zip(*self.test)
        return list(imgs), list(masks), list(idxs)

    def get_all_data(self):
        """Return every slice (train + validation + test) as three lists.
        """
        all_data = self.train + self.validation + self.test
        imgs, masks, idxs = zip(*all_data)
        return list(imgs), list(masks), list(idxs)

    def get_split_mapping(self):
        """Return {slice info: "train" | "validation" | "test"} for every slice.
        """
        mapping = {}
        for _, _, idx in self.train:
            mapping[idx] = "train"
        for _, _, idx in self.validation:
            mapping[idx] = "validation"
        for _, _, idx in self.test:
            mapping[idx] = "test"
        return mapping

    def get_cleaned_mask_stack(self, mask_stack, connectivity=1, min_size=100, smoothing_disk_size=1):
        """
        Cleans a 3D stack of 2D mask slices along axis 0 (after transposition).

        Parameters:
        - mask_stack: np.ndarray
            A 3D stack (e.g., shape: [slices, height, width]) where each slice contains labeled instances.
        - connectivity: int
            Connectivity for labeling the components (1 for 4-connectivity, 2 for 8-connectivity).
        - min_size: int
            Minimum size of instance regions to retain (in pixels).
        - smoothing_disk_size: int
            Radius of the disk-shaped structuring element used for morphological smoothing.
            Set to 0 to disable smoothing.

        Returns:
        - cleaned_stack: np.ndarray
            A cleaned version of the input stack with smoothed and filtered instances.
        """
        import numpy as np
        import tqdm
        from skimage.segmentation import find_boundaries
        from skimage.measure import label, regionprops
        from skimage.morphology import opening, closing, disk

        cleaned_stack = np.zeros_like(mask_stack, dtype=np.uint16)

        for i in tqdm.tqdm(range(mask_stack.shape[0]), desc="Cleaning 2D instance labels"):
            slice_ = mask_stack[i]

            # Remove boundaries
            boundary = find_boundaries(slice_, mode='inner')
            cleaned = slice_.copy()
            cleaned[boundary] = 0

            # Relabel the masks afterwards
            relabeled = label(cleaned, connectivity=connectivity)

            # Smooth the edges after relabeling
            smoothed = np.zeros_like(relabeled)
            if smoothing_disk_size > 0:
                selem = disk(smoothing_disk_size)


            for region in regionprops(relabeled):
                if region.area >= min_size:
                    mask = (relabeled == region.label)

                    if smoothing_disk_size > 0:
                        # Morphological smoothing
                        mask = opening(mask, selem)
                        mask = closing(mask, selem)

                    smoothed[mask] = region.label

            # Set the final mask
            cleaned_stack[i] = smoothed

        return cleaned_stack


    def save_view_aligned_stack(self, output_path_prefix):
        """
        Save the view-aligned image and mask stack to the same directory with consistent suffixes.

        Args:
            output_path_prefix (str): Base path prefix (e.g. '/path/to/patch1_XZ').

        Saves:
            - {output_path_prefix}_image.tif
            - {output_path_prefix}_mask.tif
        """
        import tifffile
        import numpy as np

        image_path = output_path_prefix + "_image.tif"
        mask_path = output_path_prefix + "_mask.tif"

        # Convert to arrays if they're still lists
        images_np = np.stack(self.images, axis=0) if isinstance(self.images, list) else self.images
        masks_np = np.stack(self.masks, axis=0) if isinstance(self.masks, list) else self.masks

        print(f"Saving image stack to: {image_path}")
        tifffile.imwrite(image_path, images_np.astype(np.float32))  # shape: (D, C, H, W) or (C, D, H, W)

        print(f"Saving mask stack to: {mask_path}")
        tifffile.imwrite(mask_path, masks_np.astype(np.uint16))     # shape: (D, H, W)

        print("Finished saving image and mask.")


    def save_data(self):
        """Write the split indices (`train_indices.npy`, `validation_indices.npy`,
        `test_indices.npy`) and a `split_info.json` to `self.save_path`.
        """
        if not hasattr(self, 'train') or not self.train:
            raise ValueError("No split data to save. Run split_data() first.")

        if self.save_path is None:
            raise ValueError("Please provide a 'save_path' to save the splits.")

        os.makedirs(self.save_path, exist_ok=True)

        np.save(os.path.join(self.save_path, "train_indices.npy"), [i for _, _, i in self.train])
        np.save(os.path.join(self.save_path, "validation_indices.npy"), [i for _, _, i in self.validation])
        np.save(os.path.join(self.save_path, "test_indices.npy"), [i for _, _, i in self.test])

        metadata = {
            "stack_name": self.stack_name,
            "view": self.view,
            "segment_range": self.segment_range,
            "image_path": self.image_path,
            "mask_path": self.mask_path,
            "spatial_split": self.spatial_split
        }

        with open(os.path.join(self.save_path, "split_info.json"), "w") as f:
            json.dump(metadata, f, indent=4)

        print(f"Split data saved to: {self.save_path}")

    def load_saved_splits(self):
        """Rebuild the three splits from the index files written by `save_data`.
        """
        if self.save_path is None:
            raise ValueError("Provide a 'save_path' to load the split data from.")

        try:
            train_ids = np.load(os.path.join(self.save_path, "train_indices.npy"))
            val_ids = np.load(os.path.join(self.save_path, "validation_indices.npy"))
            test_ids = np.load(os.path.join(self.save_path, "test_indices.npy"))
        except FileNotFoundError:
            raise FileNotFoundError("Split files not found at save_path.")

        index_to_data = {i: (img, msk, i) for img, msk, i in zip(self.images, self.masks, self.slice_indices)}

        self.train = [index_to_data[i] for i in train_ids]
        self.validation = [index_to_data[i] for i in val_ids]
        self.test = [index_to_data[i] for i in test_ids]

        print(f"Loaded splits from {self.save_path}")


    def plot_split_map(self):
        """Visualize which slice index belongs to which split as a continuous strip (legend below)."""
        split_map = self.get_split_mapping()
        all_indices = sorted(split_map.keys())

        # Map splits to integers
        split_to_label = {"train": 0, "validation": 1, "test": 2}
        label_to_color = ["tab:blue", "tab:orange", "tab:green"]

        split_labels = [split_to_label[split_map[i]] for i in all_indices]
        img = np.array(split_labels).reshape(1, -1)

        fig, ax = plt.subplots(figsize=(12, 1.5))
        ax.imshow(img, cmap=plt.matplotlib.colors.ListedColormap(label_to_color), aspect="auto")
        ax.set_yticks([])
        ax.set_xticks(np.linspace(0, len(all_indices) - 1, 10, dtype=int))
        ax.set_xlabel(f"{self.view}-view Slice Index")
        ax.set_title(f"{self.stack_name}: Slice Split Distribution")

        legend_elements = [
            Patch(facecolor=label_to_color[0], label='Train'),
            Patch(facecolor=label_to_color[1], label='Validation'),
            Patch(facecolor=label_to_color[2], label='Test'),
        ]
        fig.legend(handles=legend_elements, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.2))

        plt.tight_layout()
        plt.show()

    def evaluate_model_slicewise_metrics(self, model_path, channels=[2,1], normalize=True):
            """
            Evaluate a trained Cellpose model slice-by-slice and plot:
            Dice (line), IoU, AJI, and AP scores with split overlay.

            Returns:
                dict with keys: 'slice_idx', 'dice', 'iou', 'aji', 'ap_50', 'ap_75', 'ap_90'
            """
            from cellpose import metrics
            from matplotlib.patches import Patch
            from matplotlib.lines import Line2D

            model = models.CellposeModel(gpu=True, pretrained_model=model_path)

            all_imgs, all_masks, all_idxs = self.get_all_data()
            split_map = self.get_split_mapping()

            dice_scores, iou_scores, aji_scores = [], [], []
            ap_50_scores, ap_75_scores, ap_90_scores = [], [], []
            colors = []
            split_labels = []

            for img, gt_mask, idx in tqdm.tqdm(zip(all_imgs, all_masks, all_idxs), total=len(all_imgs), desc="Evaluating slices"):
                preds, _, _ = model.eval(
                    [img],
                    channels=channels,
                    normalize=normalize,
                    compute_masks=True,
                    diameter=None,
                    do_3D=False,
                    rescale=None,
                    cellprob_threshold=-2,
                    flow_threshold=0.4,

                )
                pred_mask = preds[0].astype(np.uint16)

                # Dice score (manual)
                intersection = np.logical_and(pred_mask > 0, gt_mask > 0).sum()
                union = (pred_mask > 0).sum() + (gt_mask > 0).sum()
                dice = (2.0 * intersection) / union if union > 0 else 0.0

                # IoU
                iou_vals, _ = metrics.mask_ious(gt_mask, pred_mask)
                mean_iou = iou_vals.mean() if len(iou_vals) > 0 else 0.0

                # AJI
                aji = metrics.aggregated_jaccard_index([gt_mask], [pred_mask])[0]

                # APs
                ap, _, _, _ = metrics.average_precision([gt_mask], [pred_mask], threshold=[0.5, 0.75, 0.9])

                # Collect
                dice_scores.append(dice)
                iou_scores.append(mean_iou)
                aji_scores.append(aji)
                ap_50_scores.append(ap[0][0])
                ap_75_scores.append(ap[0][1])
                ap_90_scores.append(ap[0][2])

                split = split_map.get(idx, "train")
                colors.append(
                    "tab:blue" if split == "train" else
                    "tab:orange" if split == "validation" else
                    "tab:green"
                )
                split_labels.append(split)

            # === Sort ===
            sorted_data = sorted(zip(all_idxs, dice_scores, iou_scores, aji_scores, ap_50_scores, ap_75_scores, ap_90_scores, colors, split_labels), key=lambda x: x[0])
            (sorted_idxs, sorted_dice, sorted_iou, sorted_aji,
            sorted_ap50, sorted_ap75, sorted_ap90, sorted_colors, sorted_splits) = zip(*sorted_data)

            # === Plot ===
            fig, ax1 = plt.subplots(figsize=(14, 5))
            ax1.plot(sorted_idxs, sorted_dice, label="Dice", color="steelblue", linewidth=1.5)

            ax2 = ax1.twinx()
            ax2.plot(sorted_idxs, sorted_iou, color="black", linewidth=1.2, label="IoU")
            ax2.plot(sorted_idxs, sorted_aji, color="purple", linestyle="--", linewidth=1.2, label="AJI")
            ax2.plot(sorted_idxs, sorted_ap50, color="red", linestyle=":", linewidth=1.0, label="AP@0.5")
            ax2.plot(sorted_idxs, sorted_ap75, color="darkorange", linestyle=":", linewidth=1.0, label="AP@0.75")
            ax2.plot(sorted_idxs, sorted_ap90, color="saddlebrown", linestyle=":", linewidth=1.0, label="AP@0.9")

            ax1.set_ylim(0, 1)
            ax2.set_ylim(0, 1)

            # Split background
            for i, c in enumerate(sorted_colors):
                ax1.axvspan(i - 0.5, i + 0.5, color=c, alpha=0.1)

            ax1.set_xlabel("Slice Index")
            ax1.set_ylabel("Dice Score")
            ax2.set_ylabel("IoU / AJI / AP Scores")
            ax1.set_title(f"{self.stack_name}: Slice-wise Segmentation Scores with Split Overlay")

            # === Legends ===
            legend_split = [
                Patch(facecolor="tab:blue", label="Train"),
                Patch(facecolor="tab:orange", label="Validation"),
                Patch(facecolor="tab:green", label="Test")
            ]
            legend_metrics = [
                Line2D([0], [0], color="steelblue", lw=1.5, label="Dice"),
                Line2D([0], [0], color="black", lw=1.2, label="IoU"),
                Line2D([0], [0], color="purple", linestyle="--", lw=1.2, label="AJI"),
                Line2D([0], [0], color="red", linestyle=":", lw=1.0, label="AP@0.5"),
                Line2D([0], [0], color="darkorange", linestyle=":", lw=1.0, label="AP@0.75"),
                Line2D([0], [0], color="saddlebrown", linestyle=":", lw=1.0, label="AP@0.9")
            ]
            fig.legend(handles=legend_split + legend_metrics, loc="lower center", ncol=7, frameon=False, bbox_to_anchor=(0.5, -0.15))

            plt.tight_layout()
            plt.show()

            return pd.DataFrame({
                "slice_idx": all_idxs,
                "split": sorted_splits,
                "dice": dice_scores,
                "iou": iou_scores,
                "aji": aji_scores,
                "ap_50": ap_50_scores,
                "ap_75": ap_75_scores,
                "ap_90": ap_90_scores
            })


In [ ]:
def prepare_dense_data_across_stacks(
    stacks,
    mode="spatial",  # "spatial" = XY view, "axial" = XZ + YZ views
    save_dir="splits_json",
    size_filter=100,
    smoothing_disk_size=1,
    relabel_masks=True,
    splits=[0.7, 0.2, 0.1]
):
    """Decompose every stack into the slices of one view set and split them (Methods §3.2-3.4).

    `mode="spatial"` uses the XY view, `mode="axial"` pools XZ and YZ. For each stack and view a
    `LoadTrainingData` is built with chunked spatial splitting in the given `splits`
    proportions (train, validation, test), a 50-slice edge buffer for axial views, and the
    per-slice label clean-up. The slice indices of each split are written to
    `<save_dir>/<stack>_<view>_splits.json`. Returns three lists of (image, mask, info)
    tuples pooled across stacks: train, validation, test.
    """
    assert mode in ["spatial", "axial"], "Mode must be either 'spatial' or 'axial'"
    os.makedirs(save_dir, exist_ok=True)

    views = ["XY"] if mode == "spatial" else ["XZ", "YZ"]
    split_train, split_val, split_test = splits

    all_train, all_val, all_test = [], [], []
    only_train = split_train == 1.0 and split_val == 0.0 and split_test == 0.0

    for stack in stacks:
        for view in views:
            print(f"\nProcessing: {stack['name']} in view {view}")

            edge_buffer = 50 if view in ["XZ", "YZ"] else 0  # 50-slice edge buffer on the axial views (Methods §3.3)

            loader = LoadTrainingData(
                stack_name=stack["name"],
                image_path=stack["image_path"],
                mask_path=stack["mask_path"],
                view=view,
                spatial_split="chunked",
                train_split=split_train,
                validation_split=split_val,
                test_split=split_test,
                relabel_masks=relabel_masks,
                size_filter=size_filter,
                smoothing_disk_size=smoothing_disk_size,
                edge_buffer=edge_buffer
            )

            train = loader.get_training()
            val, test = ([], [], []), ([], [], [])

            if not only_train:
                if split_val > 0.0:
                    val = loader.get_validation()
                if split_test > 0.0:
                    test = loader.get_test_data()

            print(f"Train: {len(train[0])}, Val: {len(val[0])}, Test: {len(test[0])}")

            split_json = {
                "stack": stack["name"],
                "view": view,
                "train": train[2],
                "val": val[2] if not only_train else [],
                "test": test[2] if not only_train else []
            }

            json_path = os.path.join(save_dir, f"{stack['name']}_{view}_splits.json")
            with open(json_path, "w") as f:
                json.dump(split_json, f, indent=4)
            print(f"Saved split file to: {json_path}")

            all_train.extend(zip(*train))
            if not only_train:
                all_val.extend(zip(*val))
                all_test.extend(zip(*test))

    return all_train, all_val, all_test

In [ ]:
data_dir = "../ground_truth/zebrafish/development/"   # the 7 development stacks (patch-N.tif + patch-N_masks.tif)

# Automatically detect patch pairs
patch_names = sorted(set(
    f.replace("_masks.tif", "").replace(".tif", "")
    for f in os.listdir(data_dir) if f.endswith(".tif")
))

# Filter out mask-only files
patch_names = [p for p in patch_names if not p.endswith("_masks")]

# Build stack list
stacks1 = []
for patch in patch_names:
    stacks1.append({
        "name": patch,
        "image_path": os.path.join(data_dir, f"{patch}.tif"),
        "mask_path": os.path.join(data_dir, f"{patch}_masks.tif")
    })

# Confirm it worked
for s in stacks1:
    print(f"{s['name']}:")
    print(f"  image: {s['image_path']}")
    print(f"  mask : {s['mask_path']}")


patch-1:
  image: data/patch-1.tif
  mask : data/patch-1_masks.tif
patch-2:
  image: data/patch-2.tif
  mask : data/patch-2_masks.tif
patch-3:
  image: data/patch-3.tif
  mask : data/patch-3_masks.tif
patch-4:
  image: data/patch-4.tif
  mask : data/patch-4_masks.tif
patch-5:
  image: data/patch-5.tif
  mask : data/patch-5_masks.tif
patch-7:
  image: data/patch-7.tif
  mask : data/patch-7_masks.tif
patch-8:
  image: data/patch-8.tif
  mask : data/patch-8_masks.tif


In [ ]:
data_dir = "../ground_truth/zebrafish/held_out/"   # the 4 held-out stacks (patch-N.tif + patch-N_mask.tif)

# Automatically detect patch pairs
patch_names = sorted(set(
    f.replace("_mask.tif", "").replace(".tif", "")
    for f in os.listdir(data_dir) if f.endswith(".tif")
))

# Filter out mask-only files
patch_names = [p for p in patch_names if not p.endswith("_mask")]

# Build stack list
stacks2 = []
for patch in patch_names:
    stacks2.append({
        "name": patch,
        "image_path": os.path.join(data_dir, f"{patch}.tif"),
        "mask_path": os.path.join(data_dir, f"{patch}_mask.tif")
    })

# Confirm it worked
for s in stacks2:
    print(f"{s['name']}:")
    print(f"  image: {s['image_path']}")
    print(f"  mask : {s['mask_path']}")


patch-10:
  image: data_holdout/patch-10.tif
  mask : data_holdout/patch-10_mask.tif
patch-14:
  image: data_holdout/patch-14.tif
  mask : data_holdout/patch-14_mask.tif
patch-6:
  image: data_holdout/patch-6.tif
  mask : data_holdout/patch-6_mask.tif
patch-9:
  image: data_holdout/patch-9.tif
  mask : data_holdout/patch-9_mask.tif


In [ ]:
stacks = stacks1 + stacks2


for s in stacks:
    print(f"{s['name']}:")
    print(f"  image: {s['image_path']}")
    print(f"  mask : {s['mask_path']}")

patch-1:
  image: data/patch-1.tif
  mask : data/patch-1_masks.tif
patch-2:
  image: data/patch-2.tif
  mask : data/patch-2_masks.tif
patch-3:
  image: data/patch-3.tif
  mask : data/patch-3_masks.tif
patch-4:
  image: data/patch-4.tif
  mask : data/patch-4_masks.tif
patch-5:
  image: data/patch-5.tif
  mask : data/patch-5_masks.tif
patch-7:
  image: data/patch-7.tif
  mask : data/patch-7_masks.tif
patch-8:
  image: data/patch-8.tif
  mask : data/patch-8_masks.tif
patch-10:
  image: data_holdout/patch-10.tif
  mask : data_holdout/patch-10_mask.tif
patch-14:
  image: data_holdout/patch-14.tif
  mask : data_holdout/patch-14_mask.tif
patch-6:
  image: data_holdout/patch-6.tif
  mask : data_holdout/patch-6_mask.tif
patch-9:
  image: data_holdout/patch-9.tif
  mask : data_holdout/patch-9_mask.tif


# Helper Functions --Export

In [ ]:
import json
import shutil
from datetime import datetime
from pathlib import Path
import tifffile


def export_split_dataset(
    dataset_root,
    dataset_name,
    train_samples,
    val_samples,
    test_samples,
    metadata
):
    """
    Export train/val/test datasets as individual TIFF image-mask pairs.
    """

    dataset_root = Path(dataset_root)
    out_root = dataset_root / dataset_name
    shutil.rmtree(out_root, ignore_errors=True)   # a rerun replaces the exported split entirely

    splits = {
        "train": train_samples,
        "val": val_samples,
        "test": test_samples
    }

    # Create directory structure
    for split in splits:
        (out_root / split / "images").mkdir(parents=True, exist_ok=True)
        (out_root / split / "masks").mkdir(parents=True, exist_ok=True)

    # Save samples
    for split, samples in splits.items():
        for img, mask, info in samples:
            stack = info["stack"]
            view = info["view"]
            slice_idx = info["slice_idx"]

            base = f"{stack}__view-{view}__slice-{slice_idx:04d}"

            img_path = out_root / split / "images" / f"{base}_img.tif"
            mask_path = out_root / split / "masks" / f"{base}_mask.tif"

            tifffile.imwrite(img_path, img)
            tifffile.imwrite(mask_path, mask.astype("uint16"))

    # Save metadata
    metadata_out = metadata.copy()
    metadata_out["created_at"] = datetime.now().isoformat()
    metadata_out["num_samples"] = {
        "train": len(train_samples),
        "val": len(val_samples),
        "test": len(test_samples),
    }

    with open(out_root / "metadata.json", "w") as f:
        json.dump(metadata_out, f, indent=2)

    print(f"Dataset exported to: {out_root}")


# Helper Functions -- Loader

In [ ]:
def load_exported_dataset(dataset_root):
    """
    Load an exported dataset snapshot from disk.
    """

    dataset_root = Path(dataset_root)

    def load_split(split):
        img_dir = dataset_root / split / "images"
        mask_dir = dataset_root / split / "masks"

        imgs, masks = [], []

        img_files = sorted(img_dir.glob("*_img.tif"))
        for img_path in img_files:
            mask_path = mask_dir / img_path.name.replace("_img.tif", "_mask.tif")

            img = tifffile.imread(img_path)
            mask = tifffile.imread(mask_path)

            imgs.append(img)
            masks.append(mask)

        return imgs, masks

    train_imgs, train_masks = load_split("train")
    val_imgs, val_masks = load_split("val")
    test_imgs, test_masks = load_split("test")

    with open(dataset_root / "metadata.json") as f:
        metadata = json.load(f)

    return (
        train_imgs, train_masks,
        val_imgs, val_masks,
        test_imgs, test_masks,
        metadata
    )


# Lateral Data

In [ ]:
train_data, val_data, test_data = prepare_dense_data_across_stacks(
    stacks=stacks,
    mode="spatial",          # ← LATERAL = XY
    save_dir="decomposed_2D/decomposed2D_benchmark-strict_splits",
    splits=[0.4, 0.3, 0.3]
)

export_split_dataset(
    dataset_root="decomposed_2D/decomposed2D_benchmark-strict_splits",
    dataset_name="lateral_dataset_v1",
    train_samples=train_data,
    val_samples=val_data,
    test_samples=test_data,
    metadata={
        "dataset_type": "lateral",
        "views": ["XY"],
        "split_type": "strict",
        "splits": [0.4, 0.3, 0.3],
        "chunk_size": 10,
        "num_patches": len(stacks)
    }
)



Processing: patch-1 in view XY
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|█████████████████████████████████████████████████████████████████████| 278/278 [00:00<00:00, 803595.12it/s]


   Train: 111, Val: 83, Test: 84
   Saved split file to: decomposed_2D/decomposed2D_benchmark-strict_splits/patch-1_XY_splits.json

Processing: patch-2 in view XY
Loaded image shape: (450, 2, 512, 512)
Loaded mask shape: (450, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 450/450 [00:00<00:00, 1290996.44it/s]


   Train: 180, Val: 135, Test: 135
   Saved split file to: decomposed_2D/decomposed2D_benchmark-strict_splits/patch-2_XY_splits.json

Processing: patch-3 in view XY
Loaded image shape: (402, 2, 512, 512)
Loaded mask shape: (402, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 402/402 [00:00<00:00, 1212156.87it/s]


... (13 outputs left out)

Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 502/502 [00:00<00:00, 1201107.02it/s]


   Train: 200, Val: 150, Test: 152
   Saved split file to: decomposed_2D/decomposed2D_benchmark-strict_splits/patch-6_XY_splits.json

Processing: patch-9 in view XY
Loaded image shape: (301, 2, 512, 512)
Loaded mask shape: (301, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 301/301 [00:00<00:00, 1089288.61it/s]


   Train: 120, Val: 90, Test: 91
   Saved split file to: decomposed_2D/decomposed2D_benchmark-strict_splits/patch-9_XY_splits.json
Dataset exported to: decomposed_2D/decomposed2D_benchmark-strict_splits/lateral_dataset_v1


In [ ]:
train_data, val_data, test_data = prepare_dense_data_across_stacks(
    stacks=stacks,
    mode="spatial",
    save_dir="decomposed_2D/decomposed2D_benchmark-lenient_splits",
    splits=[0.8, 0.2, 0.0]
)


export_split_dataset(
    dataset_root="decomposed_2D/decomposed2D_benchmark-lenient_splits",
    dataset_name="lateral_dataset_v1",
    train_samples=train_data,
    val_samples=val_data,
    test_samples=test_data,
    metadata={
        "dataset_type": "lateral",
        "views": ["XY"],
        "split_type": "lenient",
        "splits": [0.8, 0.2],
        "chunk_size": 10,
        "num_patches": len(stacks)
    }
)



Processing: patch-1 in view XY
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 278/278 [00:00<00:00, 1073680.03it/s]


   Train: 222, Val: 55, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-lenient_splits/patch-1_XY_splits.json

Processing: patch-2 in view XY
Loaded image shape: (450, 2, 512, 512)
Loaded mask shape: (450, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 450/450 [00:00<00:00, 1152985.22it/s]


   Train: 360, Val: 90, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-lenient_splits/patch-2_XY_splits.json

Processing: patch-3 in view XY
Loaded image shape: (402, 2, 512, 512)
Loaded mask shape: (402, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 402/402 [00:00<00:00, 1139263.65it/s]


... (13 outputs left out)

Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 502/502 [00:00<00:00, 1156255.14it/s]


   Train: 401, Val: 100, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-lenient_splits/patch-6_XY_splits.json

Processing: patch-9 in view XY
Loaded image shape: (301, 2, 512, 512)
Loaded mask shape: (301, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 301/301 [00:00<00:00, 1099726.05it/s]


   Train: 240, Val: 60, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-lenient_splits/patch-9_XY_splits.json
Dataset exported to: decomposed_2D/decomposed2D_benchmark-lenient_splits/lateral_dataset_v1


In [ ]:
train_data, val_data, test_data = prepare_dense_data_across_stacks(
    stacks=stacks1,
    mode="spatial",          # ← LATERAL = XY
    save_dir="decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits",
    splits=[0.4, 0.3, 0.3]
)

export_split_dataset(
    dataset_root="decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits",
    dataset_name="lateral_dataset_v1",
    train_samples=train_data,
    val_samples=val_data,
    test_samples=test_data,
    metadata={
        "dataset_type": "lateral",
        "views": ["XY"],
        "split_type": "strict",
        "splits": [0.4, 0.3, 0.3],
        "chunk_size": 10,
        "num_patches": len(stacks)
    }
)



Processing: patch-1 in view XY
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|█████████████████████████████████████████████████████████████████████| 278/278 [00:00<00:00, 774247.35it/s]


   Train: 111, Val: 83, Test: 84
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/patch-1_XY_splits.json

Processing: patch-2 in view XY
Loaded image shape: (450, 2, 512, 512)
Loaded mask shape: (450, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 450/450 [00:00<00:00, 1230402.09it/s]


   Train: 180, Val: 135, Test: 135
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/patch-2_XY_splits.json

Processing: patch-3 in view XY
Loaded image shape: (402, 2, 512, 512)
Loaded mask shape: (402, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 402/402 [00:00<00:00, 1175809.07it/s]


... (5 outputs left out)

Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 341/341 [00:00<00:00, 1256816.93it/s]


   Train: 136, Val: 102, Test: 103
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/patch-7_XY_splits.json

Processing: patch-8 in view XY
Loaded image shape: (335, 2, 512, 512)
Loaded mask shape: (335, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 335/335 [00:00<00:00, 1269278.99it/s]


   Train: 134, Val: 100, Test: 101
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/patch-8_XY_splits.json
Dataset exported to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/lateral_dataset_v1


In [ ]:
train_data, val_data, test_data = prepare_dense_data_across_stacks(
    stacks=stacks1,
    mode="spatial",
    save_dir="decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits",
    splits=[0.8, 0.2, 0.0]
)


export_split_dataset(
    dataset_root="decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits",
    dataset_name="lateral_dataset_v1",
    train_samples=train_data,
    val_samples=val_data,
    test_samples=test_data,
    metadata={
        "dataset_type": "lateral",
        "views": ["XY"],
        "split_type": "lenient",
        "splits": [0.8, 0.2],
        "chunk_size": 10,
        "num_patches": len(stacks)
    }
)



Processing: patch-1 in view XY
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 278/278 [00:00<00:00, 1122248.81it/s]


   Train: 222, Val: 55, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/patch-1_XY_splits.json

Processing: patch-2 in view XY
Loaded image shape: (450, 2, 512, 512)
Loaded mask shape: (450, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 450/450 [00:00<00:00, 1099904.90it/s]


   Train: 360, Val: 90, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/patch-2_XY_splits.json

Processing: patch-3 in view XY
Loaded image shape: (402, 2, 512, 512)
Loaded mask shape: (402, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 402/402 [00:00<00:00, 1210416.52it/s]


... (5 outputs left out)

Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 341/341 [00:00<00:00, 1273604.33it/s]


   Train: 272, Val: 68, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/patch-7_XY_splits.json

Processing: patch-8 in view XY
Loaded image shape: (335, 2, 512, 512)
Loaded mask shape: (335, 512, 512)
Cleaning 2D instance labels in XY view...


Extracting XY slices: 100%|████████████████████████████████████████████████████████████████████| 335/335 [00:00<00:00, 1284361.83it/s]


   Train: 268, Val: 67, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/patch-8_XY_splits.json
Dataset exported to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/lateral_dataset_v1


# Axial Data

In [ ]:
train_data, val_data, test_data = prepare_dense_data_across_stacks(
    stacks=stacks,
    mode="axial",             # ← AXIAL = XZ + YZ
    save_dir="decomposed_2D/decomposed2D_benchmark-strict_splits",
    splits=[0.4, 0.3, 0.3]
)

export_split_dataset(
    dataset_root="decomposed_2D/decomposed2D_benchmark-strict_splits",
    dataset_name="axial_dataset_v1",
    train_samples=train_data,
    val_samples=val_data,
    test_samples=test_data,
    metadata={
        "dataset_type": "axial",
        "views": ["XZ", "YZ"],
        "split_type": "strict",
        "splits": [0.4, 0.3, 0.3],
        "chunk_size": 10,
        "num_patches": len(stacks)
    }
)



Processing: patch-1 in view XZ
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in XZ view...


Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1147445.72it/s]


   Train: 164, Val: 123, Test: 125
   Saved split file to: decomposed_2D/decomposed2D_benchmark-strict_splits/patch-1_XZ_splits.json

Processing: patch-1 in view YZ
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in YZ view...


Extracting YZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1276257.94it/s]


   Train: 164, Val: 123, Test: 125
   Saved split file to: decomposed_2D/decomposed2D_benchmark-strict_splits/patch-1_YZ_splits.json

Processing: patch-2 in view XZ
Loaded image shape: (450, 2, 512, 512)
Loaded mask shape: (450, 512, 512)
Cleaning 2D instance labels in XZ view...


Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1242310.03it/s]


... (35 outputs left out)

Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1283843.42it/s]


   Train: 164, Val: 123, Test: 125
   Saved split file to: decomposed_2D/decomposed2D_benchmark-strict_splits/patch-9_XZ_splits.json

Processing: patch-9 in view YZ
Loaded image shape: (301, 2, 512, 512)
Loaded mask shape: (301, 512, 512)
Cleaning 2D instance labels in YZ view...


Extracting YZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1157436.87it/s]


   Train: 164, Val: 123, Test: 125
   Saved split file to: decomposed_2D/decomposed2D_benchmark-strict_splits/patch-9_YZ_splits.json
Dataset exported to: decomposed_2D/decomposed2D_benchmark-strict_splits/axial_dataset_v1


In [ ]:
train_data, val_data, test_data = prepare_dense_data_across_stacks(
    stacks=stacks,
    mode="axial",   # ← AXIAL = XZ + YZ
    save_dir="decomposed_2D/decomposed2D_benchmark-lenient_splits",
    splits=[0.8, 0.2, 0.0]
)

export_split_dataset(
    dataset_root="decomposed_2D/decomposed2D_benchmark-lenient_splits",
    dataset_name="axial_dataset_v1",
    train_samples=train_data,
    val_samples=val_data,
    test_samples=test_data,
    metadata={
        "dataset_type": "axial",
        "views": ["XZ", "YZ"],
        "split_type": "lenient",
        "splits": [0.8, 0.2],
        "chunk_size": 10,
        "num_patches": len(stacks)
    }
)


Processing: patch-1 in view XZ
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in XZ view...


Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1247691.88it/s]


   Train: 329, Val: 82, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-lenient_splits/patch-1_XZ_splits.json

Processing: patch-1 in view YZ
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in YZ view...


Extracting YZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1276257.94it/s]


   Train: 329, Val: 82, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-lenient_splits/patch-1_YZ_splits.json

Processing: patch-2 in view XZ
Loaded image shape: (450, 2, 512, 512)
Loaded mask shape: (450, 512, 512)
Cleaning 2D instance labels in XZ view...


Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1255852.65it/s]


... (35 outputs left out)

Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1251305.76it/s]


   Train: 329, Val: 82, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-lenient_splits/patch-9_XZ_splits.json

Processing: patch-9 in view YZ
Loaded image shape: (301, 2, 512, 512)
Loaded mask shape: (301, 512, 512)
Cleaning 2D instance labels in YZ view...


Extracting YZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1236089.59it/s]


   Train: 329, Val: 82, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-lenient_splits/patch-9_YZ_splits.json
Dataset exported to: decomposed_2D/decomposed2D_benchmark-lenient_splits/axial_dataset_v1


In [ ]:
train_data, val_data, test_data = prepare_dense_data_across_stacks(
    stacks=stacks1,
    mode="axial",             # ← AXIAL = XZ + YZ
    save_dir="decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits",
    splits=[0.4, 0.3, 0.3]
)

export_split_dataset(
    dataset_root="decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits",
    dataset_name="axial_dataset_v1",
    train_samples=train_data,
    val_samples=val_data,
    test_samples=test_data,
    metadata={
        "dataset_type": "axial",
        "views": ["XZ", "YZ"],
        "split_type": "strict",
        "splits": [0.4, 0.3, 0.3],
        "chunk_size": 10,
        "num_patches": len(stacks)
    }
)



Processing: patch-1 in view XZ
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in XZ view...


Extracting XZ slices: 100%|█████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 885726.93it/s]


   Train: 164, Val: 123, Test: 125
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/patch-1_XZ_splits.json

Processing: patch-1 in view YZ
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in YZ view...


Extracting YZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1319124.62it/s]


   Train: 164, Val: 123, Test: 125
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/patch-1_YZ_splits.json

Processing: patch-2 in view XZ
Loaded image shape: (450, 2, 512, 512)
Loaded mask shape: (450, 512, 512)
Cleaning 2D instance labels in XZ view...


Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1316110.62it/s]


... (19 outputs left out)

Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1237860.49it/s]


   Train: 164, Val: 123, Test: 125
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/patch-8_XZ_splits.json

Processing: patch-8 in view YZ
Loaded image shape: (335, 2, 512, 512)
Loaded mask shape: (335, 512, 512)
Cleaning 2D instance labels in YZ view...


Extracting YZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1141382.59it/s]


   Train: 164, Val: 123, Test: 125
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/patch-8_YZ_splits.json
Dataset exported to: decomposed_2D/decomposed2D_benchmark-7_patch-strict_splits/axial_dataset_v1


In [ ]:
train_data, val_data, test_data = prepare_dense_data_across_stacks(
    stacks=stacks1,
    mode="axial",   # ← AXIAL = XZ + YZ
    save_dir="decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits",
    splits=[0.8, 0.2, 0.0]
)

export_split_dataset(
    dataset_root="decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits",
    dataset_name="axial_dataset_v1",
    train_samples=train_data,
    val_samples=val_data,
    test_samples=test_data,
    metadata={
        "dataset_type": "axial",
        "views": ["XZ", "YZ"],
        "split_type": "lenient",
        "splits": [0.8, 0.2],
        "chunk_size": 10,
        "num_patches": len(stacks)
    }
)


Processing: patch-1 in view XZ
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in XZ view...


Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1428143.18it/s]


   Train: 329, Val: 82, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/patch-1_XZ_splits.json

Processing: patch-1 in view YZ
Loaded image shape: (278, 2, 512, 512)
Loaded mask shape: (278, 512, 512)
Cleaning 2D instance labels in YZ view...


Extracting YZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1321141.63it/s]


   Train: 329, Val: 82, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/patch-1_YZ_splits.json

Processing: patch-2 in view XZ
Loaded image shape: (450, 2, 512, 512)
Loaded mask shape: (450, 512, 512)
Cleaning 2D instance labels in XZ view...


Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1302225.51it/s]


... (19 outputs left out)

Extracting XZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1207584.38it/s]


   Train: 329, Val: 82, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/patch-8_XZ_splits.json

Processing: patch-8 in view YZ
Loaded image shape: (335, 2, 512, 512)
Loaded mask shape: (335, 512, 512)
Cleaning 2D instance labels in YZ view...


Extracting YZ slices: 100%|████████████████████████████████████████████████████████████████████| 412/412 [00:00<00:00, 1257680.68it/s]


   Train: 329, Val: 82, Test: 0
   Saved split file to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/patch-8_YZ_splits.json
Dataset exported to: decomposed_2D/decomposed2D_benchmark-7_patch-lenient_splits/axial_dataset_v1
